# 06 — Bias Correction and Multi-Model Ensemble Analysis
## Part 2 — Model–Reference Comparison and Bias Diagnostics

This notebook continues Part 1 and establishes the diagnostic framework required **before** applying statistical bias correction.

The analysis covers temporal alignment, calibration periods, spatial compatibility, climatological statistics, seasonal behaviour, distributions, error metrics, and spatial bias diagnostics.

The guiding principle is simple:

> Never apply a bias-correction method before establishing what the bias actually is.


## 1. Analysis Workflow

**Model + Reference**

→ coordinate and calendar checks

→ common calibration period

→ temporal alignment

→ spatial-grid compatibility

→ climatology

→ bias and error metrics

→ distribution diagnostics

→ spatial diagnostics

→ selection of an appropriate correction method


## 2. Define the Analysis Configuration

The calibration period must be explicit. Replace the example dates with the reference period defined by your study.

A separate evaluation period is recommended when independent validation is required.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

CALIBRATION_START = "1981-01-01"
CALIBRATION_END = "2010-12-31"
VARIABLE = "tasmax"

print(f"Calibration: {CALIBRATION_START} to {CALIBRATION_END}")
print(f"Variable: {VARIABLE}")


## 3. Identify the Time Coordinate

Climate files can use different coordinate names. The following helper supports common conventions without modifying the dataset.


In [ ]:
def find_time_coordinate(ds):
    candidates = ["time", "Time", "date", "datetime"]
    for name in candidates:
        if name in ds.coords:
            return name
    for name in ds.coords:
        try:
            if np.issubdtype(ds[name].dtype, np.datetime64):
                return name
        except TypeError:
            pass
    raise ValueError("No time coordinate could be identified.")


## 4. Identify Latitude and Longitude

The same principle applies to spatial coordinates. Supporting `lat/lon`, `latitude/longitude`, and `x/y` makes the diagnostic functions reusable across datasets.


In [ ]:
def find_spatial_coordinates(ds):
    lat_name = next((x for x in ["lat", "latitude", "y"] if x in ds.coords), None)
    lon_name = next((x for x in ["lon", "longitude", "x"] if x in ds.coords), None)
    if lat_name is None or lon_name is None:
        raise ValueError("Could not identify latitude and longitude.")
    return lat_name, lon_name


## 5. Inspect Time Coverage

Model and reference datasets must overlap in time. This function reports the available range without loading the complete variable into memory.


In [ ]:
def time_coverage(ds):
    t = ds[find_time_coordinate(ds)]
    return t.values[0], t.values[-1]

def print_time_coverage(ds, label="Dataset"):
    start, end = time_coverage(ds)
    print(f"{label}")
    print("-" * 50)
    print(f"Start: {start}")
    print(f"End  : {end}")


## 6. Select the Calibration Period

The calibration period is where the statistical relationship between model and reference data is estimated.

For example, if the study uses 1981–2010, both datasets should be restricted to that common period before estimating correction parameters.


In [ ]:
def select_period(ds, start, end):
    time_name = find_time_coordinate(ds)
    return ds.sel({time_name: slice(start, end)})


## 7. Temporal Alignment

Matching dates is essential. Differences can arise from calendars, leap days, timestamp conventions, missing records, or temporal resolution.

An inner join keeps only timestamps available in both datasets.


In [ ]:
def align_time(model, reference, join="inner"):
    model_time = find_time_coordinate(model)
    ref_time = find_time_coordinate(reference)

    if model_time != ref_time:
        reference = reference.rename({ref_time: model_time})

    return xr.align(model, reference, join=join)


## 8. Check Temporal Overlap

In [ ]:
def temporal_overlap(model, reference):
    model_aligned, reference_aligned = align_time(model, reference)
    time_name = find_time_coordinate(model_aligned)

    return {
        "common_count": time_name and model_aligned[time_name].size,
        "common_start": model_aligned[time_name].values[0],
        "common_end": model_aligned[time_name].values[-1],
    }

# Example:
# temporal_overlap(model_cal, reference_cal)


## 9. Spatial Grid Inspection

Before cell-by-cell bias calculations, compare the spatial extent, resolution, orientation, and coordinate values of the two datasets.


In [ ]:
def spatial_summary(ds):
    lat_name, lon_name = find_spatial_coordinates(ds)
    lat, lon = ds[lat_name], ds[lon_name]
    return {
        "lat_name": lat_name,
        "lon_name": lon_name,
        "lat_size": lat.size,
        "lon_size": lon.size,
        "lat_min": float(lat.min()),
        "lat_max": float(lat.max()),
        "lon_min": float(lon.min()),
        "lon_max": float(lon.max()),
    }


## 10. Compare Spatial Grids

In [ ]:
def compare_spatial_grids(model, reference):
    return pd.DataFrame({
        "model": spatial_summary(model),
        "reference": spatial_summary(reference),
    })

# Example:
# compare_spatial_grids(model_cal, reference_cal)


## 11. Why Regridding Matters

A grid-cell bias calculation is meaningful only when corresponding cells represent comparable locations.

If grids differ, use a documented regridding method such as nearest-neighbour, bilinear, or conservative remapping.

The appropriate method depends on the variable. Conservative remapping is particularly relevant when preserving spatial totals is important.


## 12. Grid Compatibility Test

In [ ]:
def grids_match(model, reference, tolerance=1e-8):
    mlat, mlon = find_spatial_coordinates(model)
    rlat, rlon = find_spatial_coordinates(reference)

    if model[mlat].shape != reference[rlat].shape:
        return False
    if model[mlon].shape != reference[rlon].shape:
        return False

    return (
        np.allclose(model[mlat].values, reference[rlat].values,
                    atol=tolerance, rtol=0)
        and
        np.allclose(model[mlon].values, reference[rlon].values,
                    atol=tolerance, rtol=0)
    )


## 13. Climatological Mean

The climatological mean is the first major diagnostic:

\[
\overline{X}=\frac{1}{N}\sum_{t=1}^{N}X_t
\]

Comparing model and reference climatologies reveals systematic differences in the average climate state.


In [ ]:
def climatological_mean(da, time_dim=None):
    if time_dim is None:
        time_dim = find_time_coordinate(
            da.to_dataset(name=da.name or "variable")
        )
    return da.mean(dim=time_dim, skipna=True)


## 14. Mean Bias

Mean bias is defined as:

\[
Bias = \overline{X}_{model} - \overline{X}_{reference}
\]

For temperature, a positive value means the model is warmer. For precipitation, it means the model is wetter.


In [ ]:
def mean_bias(model_da, reference_da):
    time_dim = find_time_coordinate(
        model_da.to_dataset(name=model_da.name or "model")
    )
    return (
        model_da.mean(dim=time_dim, skipna=True)
        - reference_da.mean(dim=time_dim, skipna=True)
    )


## 15. MAE and RMSE

Mean Absolute Error:

\[
MAE=\frac{1}{N}\sum|X_m-X_r|
\]

Root Mean Square Error:

\[
RMSE=\sqrt{\frac{1}{N}\sum(X_m-X_r)^2}
\]

RMSE gives greater influence to large errors; MAE is easier to interpret as an average absolute difference.


In [ ]:
def mae(model_da, reference_da, dim=None):
    if dim is None:
        dim = find_time_coordinate(
            model_da.to_dataset(name=model_da.name or "model")
        )
    return np.abs(model_da - reference_da).mean(dim=dim, skipna=True)

def rmse(model_da, reference_da, dim=None):
    if dim is None:
        dim = find_time_coordinate(
            model_da.to_dataset(name=model_da.name or "model")
        )
    return np.sqrt(((model_da - reference_da) ** 2).mean(dim=dim, skipna=True))


## 16. Temporal Correlation

Pearson correlation measures linear temporal association.

A high correlation does **not** mean that a model is unbiased. A model can reproduce temporal variability while having substantial mean, variance, or distribution bias.

Therefore correlation must be interpreted alongside bias and error metrics.


In [ ]:
def temporal_correlation(model_da, reference_da, dim=None):
    if dim is None:
        dim = find_time_coordinate(
            model_da.to_dataset(name=model_da.name or "model")
        )

    m = model_da - model_da.mean(dim=dim, skipna=True)
    r = reference_da - reference_da.mean(dim=dim, skipna=True)

    numerator = (m * r).sum(dim=dim, skipna=True)
    denominator = np.sqrt(
        (m ** 2).sum(dim=dim, skipna=True)
        * (r ** 2).sum(dim=dim, skipna=True)
    )
    return numerator / denominator


## 17. Distribution Diagnostics

Mean bias alone cannot describe model error.

Two datasets may have the same mean but different variability, skewness, tails, wet-day frequency, or extremes.

This is why distribution diagnostics are essential before selecting a quantile-based correction method.


In [ ]:
def distribution_statistics(da, time_dim=None):
    if time_dim is None:
        time_dim = find_time_coordinate(
            da.to_dataset(name=da.name or "variable")
        )

    return {
        "mean": da.mean(dim=time_dim, skipna=True),
        "std": da.std(dim=time_dim, skipna=True),
        "min": da.min(dim=time_dim, skipna=True),
        "max": da.max(dim=time_dim, skipna=True),
    }


## 18. Monthly Climatology

A realistic annual mean does not guarantee a realistic seasonal cycle.

For precipitation especially, a model may reproduce annual totals while placing precipitation in the wrong months.

Monthly climatology is therefore a key pre-correction diagnostic.


In [ ]:
def monthly_climatology(da):
    time_dim = find_time_coordinate(
        da.to_dataset(name=da.name or "variable")
    )
    return da.groupby(f"{time_dim}.month").mean(
        dim=time_dim, skipna=True
    )


## 19. Plot Monthly Climatology

This helper first reduces spatial dimensions to produce a domain-mean seasonal cycle. Spatial maps should still be inspected separately.


In [ ]:
def plot_monthly_climatology(model_da, reference_da,
                               label_model="Model",
                               label_reference="Reference",
                               title="Monthly Climatology"):
    model = monthly_climatology(model_da)
    ref = monthly_climatology(reference_da)

    for dim in list(model.dims):
        if dim != "month":
            model = model.mean(dim=dim, skipna=True)

    for dim in list(ref.dims):
        if dim != "month":
            ref = ref.mean(dim=dim, skipna=True)

    plt.figure()
    plt.plot(model.month, model, marker="o", label=label_model)
    plt.plot(ref.month, ref, marker="o", label=label_reference)
    plt.xlabel("Month")
    plt.ylabel(model_da.attrs.get("units", VARIABLE))
    plt.title(title)
    plt.xticks(range(1, 13))
    plt.legend()
    plt.tight_layout()
    plt.show()


## 20. Spatial Bias Map

For compatible grids:

\[
Bias(x,y)=\overline{X_m(x,y)}-\overline{X_r(x,y)}
\]

A spatial bias map identifies regions where systematic model errors are concentrated.


In [ ]:
def plot_spatial_field(da, title, cmap="RdBu_r"):
    plt.figure(figsize=(10, 6))
    da.plot(cmap=cmap)
    plt.title(title)
    plt.tight_layout()
    plt.show()


## 21. Model–Reference Scatter Plot

Points near the 1:1 line indicate agreement.

Systematic displacement suggests mean bias, broad scatter indicates larger errors, and nonlinear structure may indicate distributional bias.

Large datasets should be sampled before plotting.


In [ ]:
def plot_model_reference_scatter(model_da, reference_da,
                                   sample_size=10000, seed=42):
    model_values = np.asarray(model_da.values).ravel()
    ref_values = np.asarray(reference_da.values).ravel()

    valid = np.isfinite(model_values) & np.isfinite(ref_values)
    model_values = model_values[valid]
    ref_values = ref_values[valid]

    if model_values.size > sample_size:
        rng = np.random.default_rng(seed)
        idx = rng.choice(model_values.size, sample_size, replace=False)
        model_values = model_values[idx]
        ref_values = ref_values[idx]

    lower = min(model_values.min(), ref_values.min())
    upper = max(model_values.max(), ref_values.max())

    plt.figure(figsize=(7, 7))
    plt.scatter(ref_values, model_values, s=8, alpha=0.35)
    plt.plot([lower, upper], [lower, upper], linestyle="--")
    plt.xlabel("Reference")
    plt.ylabel("Model")
    plt.title("Model vs Reference")
    plt.tight_layout()
    plt.show()


## 22. Quantile Comparison

Quantiles describe different parts of the distribution:

- 0.10 — lower tail
- 0.50 — median
- 0.90 — upper tail
- 0.99 — extreme upper tail

A large mismatch across quantiles indicates that mean correction alone may be inadequate.


In [ ]:
def quantile_table(model_da, reference_da, quantiles=None):
    if quantiles is None:
        quantiles = np.array([
            0.01, 0.05, 0.10, 0.25, 0.50,
            0.75, 0.90, 0.95, 0.99
        ])

    m = np.asarray(model_da.values).ravel()
    r = np.asarray(reference_da.values).ravel()

    m = m[np.isfinite(m)]
    r = r[np.isfinite(r)]

    return pd.DataFrame({
        "quantile": quantiles,
        "model": np.quantile(m, quantiles),
        "reference": np.quantile(r, quantiles),
    })


## 23. Diagnostic Decision

Use the diagnostics to identify the dominant error:

**Mainly mean bias** → additive or linear scaling may be appropriate.

**Mean + variance bias** → variance scaling may be useful.

**Strong distributional bias** → empirical or parametric quantile mapping may be appropriate.

**Precipitation occurrence bias** → consider a method that explicitly handles wet-day frequency.

**Spatial structural differences** → evaluate regridding and resolution before statistical correction.

The method should be justified by the variable and research objective, not selected only because it is convenient.


## 24. Part 2 Summary

This part established the model–reference comparison framework:

- calibration-period selection
- temporal alignment
- spatial-grid checks
- climatological means
- mean bias
- MAE and RMSE
- temporal correlation
- distribution statistics
- monthly climatology
- spatial bias diagnostics
- model/reference scatter plots
- quantile comparison

### Next

Part 3 will implement the actual bias-correction methods:

1. additive/linear scaling
2. multiplicative scaling
3. variance scaling
4. empirical quantile mapping
5. calibration and validation separation
6. before/after correction diagnostics
